# Sprint 1 — Conhecendo os Dados
## Análise Espacial da Criminalidade no Brasil — Sinesp VDE 2015 a 2026

**Aluno(a):** Cintia Nunes  
**Curso:** Ciência da Computação — UFFS

Este notebook prepara e explora **as 12 planilhas anuais do Sinesp VDE (2015–2026)**, restritas a **10 indicadores** de crimes hediondos ou equiparados (seção 3.1).

> **Papel deste notebook no projeto:** os arquivos que ele exporta (seção 19) são a **entrada de treinamento da Sprint 2**. Por isso, além da análise exploratória, ele agora **mede e exporta a qualidade dos dados** (completude por UF/ano, população, granularidade municipal), para que o modelo da Sprint 2 não aprenda com dados que, na verdade, são "UF não reportou".

### O que mudou nesta versão (v2)

| # | Mudança | Por quê |
|---|---|---|
| 1 | **Nova seção 5.2 — Completude por UF × ano × indicador** | Havia UFs com total **zero** em indicadores de alto volume (ex.: Homicídio doloso em ES e GO, 2015). Zero ali significa "não reportou", não "sem crimes". Sem isso, essas UFs aparecem como "as mais seguras" e viram rótulo errado no treino da Sprint 2. |
| 2 | **Seção 11 reescrita:** série nacional **só com UFs confiáveis** + taxa por 100 mil | O total bruto mistura crescimento real com entrada de UFs na base; a tendência 2015→2017 estava contaminada pela cobertura. |
| 3 | **População para toda a série (2015–2026)**, com Censo 2022 e interpolação marcada; exportada em CSV | Antes só existia para o ano escolhido e falhava em 2022/2023. Agora Sprint 1 e Sprint 2 usam a **mesma** população. |
| 4 | **Seleção inicial = último ano completo (2025) e "Homicídio doloso"** | O padrão anterior abria em **2026 (parcial)** + "Estupro", o que gerava análise municipal vazia e taxas distorcidas ao rodar tudo sem clicar em "Confirmar". |
| 5 | **Seção 8.1 — Granularidade municipal por indicador** | Alguns indicadores só existem em nível de UF (ex.: Estupro em 2026 tinha 162 linhas = 27 UFs × 6 meses). Agora isso é detectado e explicado, em vez de falhar em silêncio. |
| 6 | **Qualidade dos dados ampliada** (seção 7) | Antes só contava zeros (que são ~92% das linhas e não informam nada). Agora registra linhas descartadas por motivo, fonte do valor (`total_vitima` × `total`) e consistência do ano do arquivo. |
| 7 | **Leitura mais rápida** (normalização de texto só nos eventos únicos) e `ExcelFile` aberto uma única vez | A normalização era aplicada linha a linha (~400 mil por ano). |
| 8 | **UFs com dados incompletos ficam cinza/"sem dados"** em gráficos, ranking e mapas; outliers calculados só entre UFs confiáveis | Evita conclusões erradas. |
| 9 | **Funciona fora do Colab** (planilhas em `dados/`) e **validações antes da exportação** | Reprodutibilidade para o GitHub; garante que o CSV entregue à Sprint 2 está íntegro. |

### Estratégia de memória

1. lê cada ano separadamente; 2. mantém só as colunas necessárias e só os 10 indicadores; 3. gera um resumo mensal por UF × indicador × abrangência; 4. libera o arquivo bruto; 5. usa o resumo para gráficos, taxas e mapas; 6. recarrega só o ano escolhido para a análise municipal.

## 1. Fonte oficial

**Ministério da Justiça e Segurança Pública — Sinesp VDE**

Base de Dados e Notas Metodológicas dos Gestores Estaduais — Sinesp VDE 2015 a 2026:

https://www.gov.br/mj/pt-br/assuntos/sua-seguranca/seguranca-publica/estatistica/dados-nacionais-1/base-de-dados-e-notas-metodologicas-dos-gestores-estaduais-sinesp-vde-2022-e-2023

### Variável de contagem

- `evento` → indicador; `data_referencia` → ano e mês;
- `total_vitima` → usada quando preenchida; `total` → usada quando `total_vitima` estiver vazio.

Alguns indicadores são contados por **vítimas** e outros por **ocorrências**; o notebook cria `valor` priorizando `total_vitima`. A seção 8 mostra, por indicador, qual coluna foi de fato usada.

## 2. Preparação do ambiente

In [ ]:
import sys
import subprocess
import importlib.util

pacotes = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "requests": "requests",
    "geopandas": "geopandas",
    "folium": "folium",
    "ipywidgets": "ipywidgets",
    "python_calamine": "python-calamine",
    "mapclassify": "mapclassify",
    "pyogrio": "pyogrio",
}

faltando = [
    pacote
    for modulo, pacote in pacotes.items()
    if importlib.util.find_spec(modulo) is None
]

if faltando:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *faltando]
    )

print("Ambiente preparado.")

In [ ]:
from pathlib import Path
import gc
import re
import time
import zipfile
import warnings
import unicodedata

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import geopandas as gpd
import folium
import ipywidgets as widgets

from IPython.display import display, clear_output

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

PASTA_DADOS = Path("dados")
PASTA_PROCESSADOS = PASTA_DADOS / "processados"
PASTA_RESULTADOS = Path("resultados")

for pasta in [PASTA_DADOS, PASTA_PROCESSADOS, PASTA_RESULTADOS]:
    pasta.mkdir(parents=True, exist_ok=True)

try:
    import google.colab  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

print("Pastas preparadas. Executando no Colab:", EM_COLAB)

## 3. Carregamento de todas as planilhas

- **No Colab:** execute a célula e selecione **todos os arquivos de 2015 a 2026 de uma só vez** (ou use o Google Drive).
- **Fora do Colab:** coloque as planilhas `BancoVDE AAAA.xlsx` na pasta `dados/`.

O ano é identificado automaticamente pelo nome (o arquivo de 2015 pode conter `(1)` ou `(2)`).

In [ ]:
USAR_GOOGLE_DRIVE = False
PASTA_GOOGLE_DRIVE = "/content/drive/MyDrive/Sinesp"

if EM_COLAB and USAR_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    caminhos = list(Path(PASTA_GOOGLE_DRIVE).glob("BancoVDE*.xlsx"))

elif EM_COLAB:
    from google.colab import files
    print("Selecione todas as planilhas de 2015 a 2026.")
    arquivos_enviados = files.upload()
    caminhos = [Path(nome) for nome in arquivos_enviados.keys()]
    del arquivos_enviados
    gc.collect()

else:
    caminhos = list(PASTA_DADOS.glob("BancoVDE*.xlsx"))
    print(f"Fora do Colab: procurando planilhas em '{PASTA_DADOS}/'.")

if not caminhos:
    raise RuntimeError("Nenhuma planilha foi encontrada.")


def ano_do_arquivo(caminho):
    resultado = re.search(r"(20[12]\d)", caminho.name)
    return int(resultado.group(1)) if resultado else None


arquivos_por_ano = {}

for caminho in sorted(caminhos):
    ano = ano_do_arquivo(caminho)
    if ano is None:
        print("Ignorado (ano não identificado):", caminho.name)
        continue
    if ano in arquivos_por_ano:
        print(f"ATENÇÃO: há mais de um arquivo para {ano}. "
              f"Usando '{caminho.name}' (antes: '{arquivos_por_ano[ano].name}').")
    arquivos_por_ano[ano] = caminho

anos_encontrados = sorted(arquivos_por_ano.keys())
print("Anos encontrados:", anos_encontrados)

esperados = set(range(2015, 2027))
faltando = sorted(esperados - set(anos_encontrados))

if faltando:
    print("ATENÇÃO — anos não enviados:", faltando)
else:
    print("Todas as 12 planilhas foram identificadas.")

## 3.1 Indicadores analisados

A análise foi restringida a **10 indicadores** de crimes hediondos ou equiparados. Todos os demais eventos do Sinesp VDE são descartados já na leitura.

| Indicador | Natureza |
|---|---|
| Estupro | Hediondo |
| Estupro de vulnerável | Hediondo |
| Feminicídio | Hediondo |
| Homicídio doloso | Hediondo se qualificado ou praticado em atividade típica de grupo de extermínio |
| Lesão corporal seguida de morte | Hediondo se praticado contra autoridade ou agente de segurança pública no exercício da função ou em decorrência dela |
| Morte de Agente do Estado | Entra no rol de hediondos se decorrente de homicídio qualificado ou lesão corporal contra agentes de segurança |
| Roubo seguido de morte (latrocínio) | Hediondo |
| Tentativa de feminicídio | Hediondo (a tentativa de crime hediondo mantém a natureza hedionda) |
| Tentativa de homicídio | Hediondo se a tentativa for de homicídio qualificado |
| Tráfico de drogas | Equiparado a hediondo |

> A base não distingue homicídio simples de qualificado; onde a natureza é condicional, os totais incluem todos os casos do indicador.
>
> **Atenção ao nome:** "Morte de Agente do Estado" são mortes **de** agentes (vitimização). É diferente de "Morte por intervenção de Agente do Estado" (mortes **causadas por** agentes), que **não** está entre os 10 indicadores e aparece na lista de descartados.

In [ ]:
def normalizar_texto(texto):
    """Minúsculas, sem acentos e sem espaços duplicados."""
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(texto.lower().split())


# nome padronizado -> variações aceitas (já normalizadas)
INDICADORES_ALVO = {
    "Estupro": ["estupro"],
    "Estupro de vulnerável": ["estupro de vulneravel"],
    "Feminicídio": ["feminicidio"],
    "Homicídio doloso": ["homicidio doloso"],
    "Lesão corporal seguida de morte": ["lesao corporal seguida de morte"],
    "Morte de Agente do Estado": ["morte de agente do estado"],
    "Roubo seguido de morte (latrocínio)": [
        "roubo seguido de morte (latrocinio)",
        "roubo seguido de morte",
        "latrocinio",
    ],
    "Tentativa de feminicídio": ["tentativa de feminicidio"],
    "Tentativa de homicídio": ["tentativa de homicidio"],
    "Tráfico de drogas": ["trafico de drogas"],
}

NATUREZA_INDICADOR = {
    "Estupro": "Hediondo",
    "Estupro de vulnerável": "Hediondo",
    "Feminicídio": "Hediondo",
    "Homicídio doloso": "Hediondo se qualificado ou de grupo de extermínio",
    "Lesão corporal seguida de morte": "Hediondo se contra autoridade/agente de segurança",
    "Morte de Agente do Estado": "Hediondo se homicídio qualificado ou lesão contra agente de segurança",
    "Roubo seguido de morte (latrocínio)": "Hediondo",
    "Tentativa de feminicídio": "Hediondo",
    "Tentativa de homicídio": "Hediondo se homicídio qualificado",
    "Tráfico de drogas": "Equiparado a hediondo",
}

# Indicadores em que uma UF com ZERO registros no ano (ou com meses sem nenhum registro)
# quase certamente NÃO reportou, em vez de não ter tido casos. Usado na seção 5.2 e na Sprint 2.
# Os demais (feminicídio, latrocínio, morte de agente etc.) têm zeros legítimos em UFs pequenas.
INDICADORES_ALTO_VOLUME = {
    "Homicídio doloso",
    "Tentativa de homicídio",
    "Tráfico de drogas",
    "Estupro",
    "Estupro de vulnerável",
}

INDICADOR_DESTAQUE = "Homicídio doloso"   # indicador-alvo padrão da Sprint 2

MAPA_ALIASES = {
    alias: nome
    for nome, aliases in INDICADORES_ALVO.items()
    for alias in aliases
}

# preenchido durante a leitura, para conferência
EVENTOS_DESCARTADOS = set()

print("Indicadores selecionados:", len(INDICADORES_ALVO))
for nome in INDICADORES_ALVO:
    print("-", nome)

## 4. Funções de preparação

A leitura usa `calamine` (mais rápida que `openpyxl` para arquivos grandes) e só carrega as colunas necessárias.

A função `ler_ano` também preenche um dicionário `info` com contagens de **linhas descartadas por motivo** e da **fonte do valor** — usado na seção 7.

In [ ]:
COLUNAS_NECESSARIAS = [
    "uf", "municipio", "evento", "data_referencia",
    "total_vitima", "total", "abrangencia",
]

MESES = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro",
}

UFS_VALIDAS = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF",
    "ES", "GO", "MA", "MT", "MS", "MG", "PA",
    "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
    "RO", "RR", "SC", "SP", "SE", "TO",
}

MUNICIPIOS_INVALIDOS = {"", "NAN", "NÃO INFORMADO", "NAO INFORMADO"}


def ler_ano(ano, incluir_municipio=True, info=None):
    """Lê a planilha de um ano, filtra os 10 indicadores e padroniza as colunas.

    Se `info` (dict) for passado, é preenchido com contagens de qualidade.
    """
    caminho = arquivos_por_ano[int(ano)]
    info = {} if info is None else info

    try:
        excel = pd.ExcelFile(caminho, engine="calamine")
        planilha = str(ano) if str(ano) in excel.sheet_names else excel.sheet_names[0]
        colunas = [c for c in COLUNAS_NECESSARIAS if incluir_municipio or c != "municipio"]
        df = pd.read_excel(excel, sheet_name=planilha, usecols=colunas)
    except Exception as erro:
        raise RuntimeError(f"Erro ao ler {caminho.name}: {erro}")

    info["linhas_lidas"] = len(df)

    # Indicador: normaliza só os nomes ÚNICOS de evento (muito mais rápido que linha a linha)
    eventos = df["evento"].astype(str)
    mapa_evento = {e: MAPA_ALIASES.get(normalizar_texto(e)) for e in eventos.unique()}
    df["indicador"] = eventos.map(mapa_evento)

    EVENTOS_DESCARTADOS.update(
        eventos[df["indicador"].isna()].str.strip().unique()
    )
    df = df[df["indicador"].notna()].copy()
    info["linhas_indicadores"] = len(df)

    # Data, UF e valor — contando o que será descartado e por quê
    df["data_referencia"] = pd.to_datetime(df["data_referencia"], errors="coerce")
    info["descartadas_data_invalida"] = int(df["data_referencia"].isna().sum())

    df["uf"] = df["uf"].astype(str).str.strip().str.upper()
    info["descartadas_uf_invalida"] = int((~df["uf"].isin(UFS_VALIDAS)).sum())

    total_vitima = pd.to_numeric(df["total_vitima"], errors="coerce")
    total = pd.to_numeric(df["total"], errors="coerce")
    df["valor"] = total_vitima.combine_first(total)

    info["linhas_valor_de_total_vitima"] = int(total_vitima.notna().sum())
    info["linhas_valor_de_total"] = int((total_vitima.isna() & total.notna()).sum())
    info["vitima_zero_com_total_positivo"] = int(((total_vitima == 0) & (total > 0)).sum())
    info["descartadas_valor_nulo"] = int(df["valor"].isna().sum())
    info["fonte_por_indicador"] = (
        total_vitima.notna().groupby(df["indicador"]).mean()
    )

    df = df.dropna(subset=["data_referencia", "valor"])
    df = df[df["uf"].isin(UFS_VALIDAS)].copy()

    if df.empty:
        raise RuntimeError(f"{caminho.name}: nenhum registro válido após os filtros.")

    df["ano"] = df["data_referencia"].dt.year.astype(int)
    df["mes_numero"] = df["data_referencia"].dt.month.astype(int)
    df["mes"] = df["mes_numero"].map(MESES)

    # O ano das datas deve coincidir com o ano do nome do arquivo
    info["linhas_ano_diferente_do_arquivo"] = int((df["ano"] != int(ano)).sum())

    df["abrangencia"] = df["abrangencia"].fillna("Não informado").astype(str).str.strip()

    if incluir_municipio:
        df["municipio"] = df["municipio"].fillna("").astype(str).str.strip()

    info["valores_negativos"] = int((df["valor"] < 0).sum())

    return df

## 5. Processamento das 12 planilhas

Esta é a etapa mais demorada. Cada planilha é carregada **uma única vez**, resumida e removida da memória.

Além do resumo mensal, esta etapa produz: cobertura temporal, qualidade por ano, **granularidade municipal por indicador** e **fonte do valor** (`total_vitima` × `total`).

O município é lido aqui só para medir a granularidade; o resumo salvo continua sem município.

In [ ]:
resumos = []
cobertura = []
qualidade = []
granularidade = []
fontes_valor = []

for ano in anos_encontrados:
    print(f"Processando {ano}...")

    info = {}
    df_ano = ler_ano(ano, incluir_municipio=True, info=info)

    cobertura.append({
        "ano": ano,
        "arquivo": arquivos_por_ano[ano].name,
        "registros_brutos": len(df_ano),
        "mes_inicial": int(df_ano["mes_numero"].min()),
        "mes_final": int(df_ano["mes_numero"].max()),
        "indicadores": int(df_ano["indicador"].nunique()),
        "ufs": int(df_ano["uf"].nunique()),
        "abrangencias": int(df_ano["abrangencia"].nunique()),
    })

    qualidade.append({
        "ano": ano,
        **{k: v for k, v in info.items() if not isinstance(v, pd.Series)},
        "valores_zero": int((df_ano["valor"] == 0).sum()),
        "uf_unicas": int(df_ano["uf"].nunique()),
        "indicadores_unicos": int(df_ano["indicador"].nunique()),
    })

    # Granularidade municipal: quantos municípios distintos cada indicador tem no ano
    mun_valido = ~df_ano["municipio"].str.upper().isin(MUNICIPIOS_INVALIDOS)
    g = pd.DataFrame({
        "linhas": df_ano.groupby("indicador").size(),
        "linhas_com_municipio": mun_valido.groupby(df_ano["indicador"]).sum(),
        "municipios_distintos": df_ano[mun_valido].groupby("indicador")["municipio"].nunique(),
    }).fillna(0).astype(int).reset_index()
    g.insert(0, "ano", ano)
    granularidade.append(g)

    fonte = info["fonte_por_indicador"].rename("pct_linhas_com_total_vitima").reset_index()
    fonte.insert(0, "ano", ano)
    fontes_valor.append(fonte)

    resumo_ano = (
        df_ano
        .groupby(
            ["ano", "mes_numero", "mes", "uf", "indicador", "abrangencia"],
            as_index=False,
            dropna=False,
        )["valor"]
        .sum()
    )
    resumos.append(resumo_ano)

    del df_ano, resumo_ano, mun_valido
    gc.collect()

resumo_sinesp = pd.concat(resumos, ignore_index=True)
cobertura = pd.DataFrame(cobertura).sort_values("ano").reset_index(drop=True)
qualidade = pd.DataFrame(qualidade).sort_values("ano").reset_index(drop=True)
granularidade = pd.concat(granularidade, ignore_index=True)
fontes_valor = pd.concat(fontes_valor, ignore_index=True)

# Meses cobertos por ano (12 = ano completo). Usado em toda a análise de completude.
MESES_ESPERADOS = cobertura.set_index("ano")["mes_final"].to_dict()
ANOS_COMPLETOS = [a for a, m in MESES_ESPERADOS.items() if m == 12]
ANOS_PARCIAIS = [a for a, m in MESES_ESPERADOS.items() if m < 12]

del resumos
gc.collect()

print()
print("Processamento concluído.")
print("Dimensão do resumo:", resumo_sinesp.shape)
print("Anos completos:", ANOS_COMPLETOS)
print("Anos parciais :", ANOS_PARCIAIS)

### 5.1 Verificação dos 10 indicadores

Total de cada indicador por ano. Se uma linha estiver zerada ou ausente em algum ano, o nome do evento nessa planilha pode ser diferente do esperado — confira a lista de descartados e ajuste os aliases em `INDICADORES_ALVO`.

> Os totais abaixo **somam todas as UFs, inclusive as que não reportaram**. Veja a seção 5.2 antes de interpretar tendências.

In [ ]:
verificacao = (
    resumo_sinesp
    .pivot_table(
        index="indicador", columns="ano", values="valor",
        aggfunc="sum", fill_value=0,
    )
    .reindex(list(INDICADORES_ALVO.keys()))
    .fillna(0)
)

display(verificacao.astype(int))

ausentes = [
    nome for nome in INDICADORES_ALVO
    if nome not in set(resumo_sinesp["indicador"])
]

if ausentes:
    print("ATENÇÃO — indicadores NÃO encontrados em nenhuma planilha:")
    for nome in ausentes:
        print("-", nome)
else:
    print("Os 10 indicadores foram encontrados.")

print()
print(f"Eventos descartados (fora da seleção): {len(EVENTOS_DESCARTADOS)}")
display(
    pd.Series(sorted(EVENTOS_DESCARTADOS), name="evento_descartado").to_frame()
)

### 5.2 Completude por UF × ano × indicador  *(nova — essencial para a Sprint 2)*

**Problema:** nos arquivos, uma UF que ainda não enviava um indicador aparece com **zero** (ou sem linhas). Somado e dividido pela população, isso vira "taxa 0", ou seja, a UF parece a mais segura do país. Na Sprint 2 isso viraria rótulo "risco Baixo" **falso**.

**Critério (por UF, ano e indicador):**

- `meses_com_registro` = nº de meses com valor > 0 **na abrangência Estadual** (`ABRANGENCIA_COMPLETUDE`; PF e PRF atuam em todas as UFs e escondem o não-reporte estadual);
- `meses_esperados` = 12 (ou os meses cobertos, em ano parcial);
- para os indicadores de **alto volume** (`INDICADORES_ALTO_VOLUME`), se `meses_com_registro < meses_esperados - MESES_SEM_REGISTRO_TOLERADOS` o dado é marcado **`suspeito`** (provável não-reporte). Por padrão a tolerância é **0** (rigoroso); aumente para 1 ou 2 se a lista de suspeitas incluir UFs pequenas com um mês legitimamente sem casos;
- para os demais, zeros são plausíveis (poucos casos) e **não** são marcados.

`confiavel = não suspeito`. Essa coluna é usada nos gráficos, mapas e exportada para a Sprint 2.

In [ ]:
# Quantos meses sem registro ainda são tolerados antes de marcar a UF como suspeita.
# 0 = rigoroso (padrão). Use 1 ou 2 se UFs pequenas aparecerem como falso positivo.
MESES_SEM_REGISTRO_TOLERADOS = 0

# A completude é medida SÓ na abrangência Estadual: a Polícia Federal e a PRF atuam em todas as UFs
# e "mascaram" a ausência de reporte da polícia estadual (ex.: tráfico de drogas em ES e TO, 2015-2017,
# só tinha registros federais). Use None para somar todas as abrangências.
ABRANGENCIA_COMPLETUDE = "Estadual"

if ABRANGENCIA_COMPLETUDE is not None and ABRANGENCIA_COMPLETUDE not in set(resumo_sinesp["abrangencia"]):
    print(f"Abrangência '{ABRANGENCIA_COMPLETUDE}' não existe nos dados; usando todas.")
    ABRANGENCIA_COMPLETUDE = None

base_completude = (
    resumo_sinesp if ABRANGENCIA_COMPLETUDE is None
    else resumo_sinesp[resumo_sinesp["abrangencia"] == ABRANGENCIA_COMPLETUDE]
)

mensal = (
    base_completude
    .groupby(["ano", "mes_numero", "uf", "indicador"], as_index=False)["valor"]
    .sum()
)

meses_reg = (
    mensal
    .groupby(["ano", "uf", "indicador"])["valor"]
    .agg(meses_com_registro=lambda s: int((s > 0).sum()))
    .reset_index()
)
valor_total = (
    resumo_sinesp  # valor anual com TODAS as abrangências
    .groupby(["ano", "uf", "indicador"], as_index=False)["valor"].sum()
    .rename(columns={"valor": "valor_anual"})
)
completude = valor_total.merge(meses_reg, on=["ano", "uf", "indicador"], how="outer")

# Grade completa ano × UF × indicador: combinações ausentes viram zero meses com registro
grade = pd.MultiIndex.from_product(
    [anos_encontrados, sorted(UFS_VALIDAS), list(INDICADORES_ALVO)],
    names=["ano", "uf", "indicador"],
).to_frame(index=False)

completude = (
    grade.merge(completude, on=["ano", "uf", "indicador"], how="left")
    .fillna({"valor_anual": 0, "meses_com_registro": 0})
)
completude["meses_com_registro"] = completude["meses_com_registro"].astype(int)
completude["meses_esperados"] = completude["ano"].map(MESES_ESPERADOS).astype(int)
completude["alto_volume"] = completude["indicador"].isin(INDICADORES_ALTO_VOLUME)
completude["suspeito"] = (
    completude["alto_volume"]
    & (completude["meses_com_registro"] < completude["meses_esperados"] - MESES_SEM_REGISTRO_TOLERADOS)
)
completude["confiavel"] = ~completude["suspeito"]

ordem_alto = [i for i in INDICADORES_ALVO if i in INDICADORES_ALTO_VOLUME]

suspeitos_por_ano = (
    completude[completude["suspeito"]]
    .pivot_table(index="indicador", columns="ano", values="uf", aggfunc="count", fill_value=0)
    .reindex(index=ordem_alto, columns=anos_encontrados)
    .fillna(0)
    .astype(int)
)

print("Nº de UFs com dado SUSPEITO (provável não-reporte), por indicador de alto volume e ano:")
display(suspeitos_por_ano)

detalhe = (
    completude[completude["suspeito"]]
    .groupby(["indicador", "ano"])["uf"]
    .apply(lambda s: ", ".join(sorted(s)))
    .reset_index(name="ufs_suspeitas")
)
print("\nQuais UFs:")
display(detalhe)

In [ ]:
def mapa_completude(indicador):
    """Heatmap UF × ano: fração dos meses esperados com algum registro."""
    d = (
        completude[completude["indicador"] == indicador]
        .assign(fracao=lambda x: x["meses_com_registro"] / x["meses_esperados"])
        .pivot(index="uf", columns="ano", values="fracao")
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(d.values, aspect="auto", vmin=0, vmax=1, cmap="RdYlGn")
    ax.set_xticks(range(len(d.columns)))
    ax.set_xticklabels(d.columns, rotation=45)
    ax.set_yticks(range(len(d.index)))
    ax.set_yticklabels(d.index)
    ax.set_title(f"Completude mensal — {indicador}\n(verde = todos os meses com registro; vermelho = sem registro)")
    fig.colorbar(im, ax=ax, label="fração de meses com registro")
    plt.tight_layout()
    plt.show()


mapa_completude(INDICADOR_DESTAQUE)

## 6. Cobertura temporal das planilhas

In [ ]:
display(cobertura)

cobertura["cobertura_meses"] = cobertura["mes_final"].apply(lambda x: MESES.get(x, x))

parciais = cobertura[cobertura["mes_final"] < 12]

if not parciais.empty:
    print("ATENÇÃO: existem anos com cobertura parcial:")
    display(parciais[["ano", "mes_inicial", "mes_final", "cobertura_meses"]])

## 7. Qualidade dos dados

Para cada ano, quantas linhas foram lidas, quantas pertencem aos 10 indicadores, quantas foram descartadas e por quê, e de onde veio o valor.

> A contagem de **zeros** deixou de ser o foco: a maior parte das linhas da base bruta é zero por construção (grade município × mês × evento), então esse número sozinho não indica problema. O que importa é a **completude por UF** (seção 5.2).

Colunas de atenção: `linhas_ano_diferente_do_arquivo` (deve ser 0) e `vitima_zero_com_total_positivo` (casos em que `total_vitima = 0` mas `total > 0`; nesses, o valor usado é 0).

In [ ]:
display(qualidade)

problemas = qualidade[
    (qualidade["linhas_ano_diferente_do_arquivo"] > 0)
    | (qualidade["valores_negativos"] > 0)
    | (qualidade["descartadas_data_invalida"] > 0)
    | (qualidade["descartadas_uf_invalida"] > 0)
]

if problemas.empty:
    print("Sem linhas descartadas por data/UF inválida, sem valores negativos e anos dos arquivos consistentes.")
else:
    print("ATENÇÃO — conferir os anos abaixo:")
    display(problemas)

print("\nRegistros no resumo:", len(resumo_sinesp))
print("Anos:", sorted(int(a) for a in resumo_sinesp["ano"].unique()))
print("Indicadores diferentes:", resumo_sinesp["indicador"].nunique())

## 8. Lista dos indicadores e fonte do valor

Mostra os 10 indicadores, sua natureza e **qual coluna de contagem foi usada** (`total_vitima` ou `total`). Se, num mesmo indicador, a fração variar muito entre anos, a unidade de medida mudou (vítimas ↔ ocorrências) e a série deve ser lida com cautela.

In [ ]:
indicadores_gerais = (
    resumo_sinesp["indicador"]
    .drop_duplicates().sort_values().reset_index(drop=True).to_frame("indicador")
)
indicadores_gerais["natureza"] = indicadores_gerais["indicador"].map(NATUREZA_INDICADOR)
indicadores_gerais["alto_volume"] = indicadores_gerais["indicador"].isin(INDICADORES_ALTO_VOLUME)

display(indicadores_gerais)
print("Total de indicadores analisados:", len(indicadores_gerais))

print("\nFração das linhas em que o valor veio de `total_vitima` (o restante veio de `total`):")
display(
    fontes_valor
    .pivot(index="indicador", columns="ano", values="pct_linhas_com_total_vitima")
    .round(2)
)

### 8.1 Granularidade municipal por indicador  *(nova)*

Nem todo indicador tem detalhamento por município. A tabela mostra quantos **municípios distintos** cada indicador possui por ano. Onde o valor é 0, o indicador só existe em nível de **UF** — e a análise municipal (seção 15) não se aplica a ele.

In [ ]:
gran_pivot = (
    granularidade
    .pivot(index="indicador", columns="ano", values="municipios_distintos")
    .reindex(list(INDICADORES_ALVO))
    .fillna(0).astype(int)
)
display(gran_pivot)

INDICADORES_COM_MUNICIPIO = sorted(
    gran_pivot.index[gran_pivot.max(axis=1) >= 100]
)
INDICADORES_SO_UF = sorted(set(INDICADORES_ALVO) - set(INDICADORES_COM_MUNICIPIO))

print("Com detalhamento municipal:", INDICADORES_COM_MUNICIPIO)
print("Somente nível de UF       :", INDICADORES_SO_UF)

## 9. População por UF e ano (SIDRA/IBGE) — série completa  *(reescrita)*

Antes, a população era consultada apenas para o ano escolhido e falhava em 2022 e 2023. Agora a série **2015–2026** é montada de uma vez:

1. **Estimativas** da tabela SIDRA 6579 (variável 9324);
2. **Censo 2022** (tabela 4714, variável 93) para 2022, quando a 6579 não traz o ano;
3. anos ainda ausentes (ex.: 2023) são **interpolados linearmente** por UF; pontas repetem o valor mais próximo. Essas linhas ficam marcadas em `populacao_interpolada` / `fonte_populacao`.

O resultado é exportado em `populacao_uf_ano.csv` e **reutilizado na Sprint 2**, garantindo a mesma população nas duas etapas. Há *cache*: se o arquivo já existir e cobrir todos os anos, não consulta o IBGE de novo.

> Caso o IBGE esteja fora do ar, o notebook continua com valores absolutos (as taxas ficam `NaN`).

In [ ]:
SIDRA_TABELA_POP = 6579
SIDRA_VARIAVEL_POP = 9324
SIDRA_TABELA_CENSO = 4714
SIDRA_VARIAVEL_CENSO = 93

USAR_CACHE_POPULACAO = True
ARQUIVO_POPULACAO = PASTA_PROCESSADOS / "populacao_uf_ano.csv"

CODIGO_UF = {
    11: "RO", 12: "AC", 13: "AM", 14: "RR", 15: "PA", 16: "AP", 17: "TO",
    21: "MA", 22: "PI", 23: "CE", 24: "RN", 25: "PB", 26: "PE", 27: "AL",
    28: "SE", 29: "BA", 31: "MG", 32: "ES", 33: "RJ", 35: "SP", 41: "PR",
    42: "SC", 43: "RS", 50: "MS", 51: "MT", 52: "GO", 53: "DF",
}


def _consultar_sidra(url, tentativas=3, pausa=3):
    ultimo_erro = None
    for _ in range(tentativas):
        try:
            resposta = requests.get(url, timeout=60)
            resposta.raise_for_status()
            return resposta.json()
        except Exception as erro:
            ultimo_erro = erro
            time.sleep(pausa)
    print(f"Falha ao consultar o SIDRA ({ultimo_erro}) -> {url}")
    return None


def _ler_sidra_uf(dados, ano):
    linhas = []
    for registro in (dados or [])[1:]:
        try:
            codigo = int(registro["D1C"])
            populacao = float(registro["V"])
        except (KeyError, ValueError, TypeError):
            continue
        uf = CODIGO_UF.get(codigo)
        if uf is not None:
            linhas.append({"uf": uf, "ano": int(ano), "populacao": populacao})
    df = pd.DataFrame(linhas, columns=["uf", "ano", "populacao"])
    # Mesmo vazio (ano sem dados no SIDRA), mantém os tipos corretos para o merge/interpolação
    return df.astype({"uf": "object", "ano": "int64", "populacao": "float64"})


def montar_populacao(anos, ufs):
    partes = []

    for ano in anos:
        print(f"Consultando população estimada de {ano}...")
        dados = _consultar_sidra(
            "https://apisidra.ibge.gov.br/values/"
            f"t/{SIDRA_TABELA_POP}/n3/all/v/{SIDRA_VARIAVEL_POP}/p/{ano}"
        )
        p = _ler_sidra_uf(dados, ano)
        p["fonte_populacao"] = f"SIDRA {SIDRA_TABELA_POP} (estimativa)"
        partes.append(p)

    pop = pd.concat(partes, ignore_index=True)

    if 2022 in anos and 2022 not in set(pop["ano"]):
        print("Estimativa de 2022 ausente; consultando o Censo 2022...")
        dados = _consultar_sidra(
            "https://apisidra.ibge.gov.br/values/"
            f"t/{SIDRA_TABELA_CENSO}/n3/all/v/{SIDRA_VARIAVEL_CENSO}/p/2022"
        )
        p = _ler_sidra_uf(dados, 2022)
        p["fonte_populacao"] = f"SIDRA {SIDRA_TABELA_CENSO} (Censo 2022)"
        pop = pd.concat([pop, p], ignore_index=True)

    grade_pop = pd.MultiIndex.from_product(
        [sorted(ufs), sorted(anos)], names=["uf", "ano"]
    ).to_frame(index=False)

    pop = (
        grade_pop.merge(pop, on=["uf", "ano"], how="left")
        .sort_values(["uf", "ano"])
        .reset_index(drop=True)
    )
    pop["populacao_interpolada"] = pop["populacao"].isna()
    pop["populacao"] = pop.groupby("uf")["populacao"].transform(
        lambda s: s.interpolate(method="linear", limit_direction="both")
    )
    pop["fonte_populacao"] = pop["fonte_populacao"].fillna("interpolada/repetida")
    return pop


pop_uf_ano = None

if USAR_CACHE_POPULACAO and ARQUIVO_POPULACAO.exists():
    cache = pd.read_csv(ARQUIVO_POPULACAO, encoding="utf-8-sig")
    cache = cache[cache["ano"].isin(anos_encontrados)]
    if set(anos_encontrados) <= set(cache["ano"]) and cache["populacao"].notna().all():
        pop_uf_ano = cache.reset_index(drop=True)
        print("População carregada do cache:", ARQUIVO_POPULACAO)

if pop_uf_ano is None:
    pop_uf_ano = montar_populacao(anos_encontrados, UFS_VALIDAS)

if pop_uf_ano["populacao"].isna().any():
    print("ATENÇÃO: população indisponível; taxas por 100 mil NÃO serão calculadas (valores absolutos apenas).")
else:
    print("\nOrigem da população por ano:")
    display(
        pop_uf_ano.groupby("ano")["fonte_populacao"]
        .agg(lambda s: ", ".join(sorted(set(s))))
        .to_frame()
    )

display(pop_uf_ano.head())

## 10. Escolha do ano, indicador e abrangência

Escolha o ano, o indicador e a abrangência e clique em **Confirmar seleção**.

**Padrão (se você apenas executar tudo):** último **ano completo**, indicador **Homicídio doloso**, abrangência **Todas**. Anos parciais aparecem marcados na lista.

In [ ]:
def rotulo_ano(ano):
    m = MESES_ESPERADOS[ano]
    return f"{ano}" if m == 12 else f"{ano} (parcial: jan–{MESES[m][:3].lower()})"


def rotulo_periodo(ano):
    return rotulo_ano(int(ano))


def indicadores_do_ano(ano):
    return sorted(
        resumo_sinesp.loc[resumo_sinesp["ano"] == int(ano), "indicador"].dropna().unique().tolist()
    )


def abrangencias_do_ano(ano):
    valores = sorted(
        resumo_sinesp.loc[resumo_sinesp["ano"] == int(ano), "abrangencia"].dropna().unique().tolist()
    )
    return ["Todas"] + valores


ANO_PADRAO = max(ANOS_COMPLETOS) if ANOS_COMPLETOS else max(anos_encontrados)

seletor_ano = widgets.Dropdown(
    options=[(rotulo_ano(a), a) for a in anos_encontrados],
    value=ANO_PADRAO,
    description="Ano:",
)

indicadores_iniciais = indicadores_do_ano(seletor_ano.value)

seletor_indicador = widgets.Dropdown(
    options=indicadores_iniciais,
    value=INDICADOR_DESTAQUE if INDICADOR_DESTAQUE in indicadores_iniciais else indicadores_iniciais[0],
    description="Indicador:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="750px"),
)

seletor_abrangencia = widgets.Dropdown(
    options=abrangencias_do_ano(seletor_ano.value),
    value="Todas",
    description="Abrangência:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="500px"),
)

botao_confirmar = widgets.Button(description="Confirmar seleção", button_style="success")
saida_selecao = widgets.Output()

# Valores padrão: valem mesmo que o botão não seja clicado
ANO_ANALISE = int(seletor_ano.value)
INDICADOR_ANALISE = seletor_indicador.value
ABRANGENCIA_ANALISE = seletor_abrangencia.value


def atualizar_opcoes(change):
    ano = int(change["new"])
    anterior = seletor_indicador.value
    indicadores = indicadores_do_ano(ano)
    seletor_indicador.options = indicadores
    if indicadores:
        seletor_indicador.value = anterior if anterior in indicadores else indicadores[0]
    seletor_abrangencia.options = abrangencias_do_ano(ano)
    seletor_abrangencia.value = "Todas"


def confirmar_selecao(_):
    global ANO_ANALISE, INDICADOR_ANALISE, ABRANGENCIA_ANALISE
    ANO_ANALISE = int(seletor_ano.value)
    INDICADOR_ANALISE = seletor_indicador.value
    ABRANGENCIA_ANALISE = seletor_abrangencia.value

    with saida_selecao:
        clear_output()
        print("Seleção confirmada.")
        print("Ano:", rotulo_ano(ANO_ANALISE))
        print("Indicador:", INDICADOR_ANALISE)
        print("Abrangência:", ABRANGENCIA_ANALISE)
        if MESES_ESPERADOS[ANO_ANALISE] < 12:
            print("\nAVISO: ano PARCIAL. Totais e taxas cobrem só parte do ano e não são "
                  "comparáveis a anos completos.")
        print("\nAgora execute as células abaixo para atualizar a análise.")


seletor_ano.observe(atualizar_opcoes, names="value")
botao_confirmar.on_click(confirmar_selecao)

display(widgets.VBox([seletor_ano, seletor_indicador, seletor_abrangencia, botao_confirmar, saida_selecao]))

print(f"Seleção atual -> Ano: {rotulo_ano(ANO_ANALISE)} | Indicador: {INDICADOR_ANALISE} | Abrangência: {ABRANGENCIA_ANALISE}")

## 11. Comparação entre os 10 indicadores no ano selecionado

Atenção às escalas: Tráfico de drogas é ordens de grandeza maior que Morte de Agente do Estado, e alguns indicadores contam **vítimas** e outros **ocorrências** (seção 8). A coluna `ufs_com_dado_suspeito` mostra quantas UFs têm provável não-reporte naquele indicador/ano.

In [ ]:
dados_ano = resumo_sinesp[resumo_sinesp["ano"] == ANO_ANALISE].copy()

if ABRANGENCIA_ANALISE != "Todas":
    dados_ano = dados_ano[dados_ano["abrangencia"] == ABRANGENCIA_ANALISE].copy()

totais_indicadores_ano = (
    dados_ano.groupby("indicador", as_index=False)["valor"].sum()
    .sort_values("valor", ascending=False)
)

suspeitas_ano = (
    completude[(completude["ano"] == ANO_ANALISE) & completude["suspeito"]]
    .groupby("indicador").size().rename("ufs_com_dado_suspeito")
)
totais_indicadores_ano = (
    totais_indicadores_ano.merge(suspeitas_ano, on="indicador", how="left")
    .fillna({"ufs_com_dado_suspeito": 0})
    .astype({"ufs_com_dado_suspeito": int})
)

display(totais_indicadores_ano)

dados_grafico = totais_indicadores_ano.sort_values("valor", ascending=True)

ax = dados_grafico.plot(
    x="indicador", y="valor", kind="barh",
    figsize=(13, max(5, len(dados_grafico) * 0.5)), legend=False,
)
for barra, v in zip(ax.patches, dados_grafico["valor"]):
    ax.text(barra.get_width(), barra.get_y() + barra.get_height() / 2, f" {int(v):,}".replace(",", "."), va="center")

ax.set_title(f"Indicadores selecionados — {rotulo_periodo(ANO_ANALISE)}")
ax.set_xlabel("Total registrado")
ax.set_ylabel("Indicador")
plt.tight_layout()
plt.show()

## 12. Evolução do indicador selecionado entre 2015 e 2026  *(reescrita)*

Dois painéis:

- **Esquerda:** total nacional **somando todas as UFs** (tracejado) × **somando só as UFs confiáveis** (linha cheia). A diferença entre as duas é o efeito da cobertura da base, não da criminalidade.
- **Direita:** **taxa nacional por 100 mil habitantes**, calculada só com as UFs confiáveis (soma dos casos ÷ soma da população dessas mesmas UFs). Anos parciais ficam de fora desta taxa, pois dividiriam 6 meses de casos por uma população anual.

Marcadores vazados ("x") indicam ano parcial.

In [ ]:
base = resumo_sinesp[resumo_sinesp["indicador"] == INDICADOR_ANALISE].copy()

if ABRANGENCIA_ANALISE != "Todas":
    base = base[base["abrangencia"] == ABRANGENCIA_ANALISE].copy()

conf = completude.loc[
    completude["indicador"] == INDICADOR_ANALISE, ["ano", "uf", "confiavel"]
]

base = base.merge(conf, on=["ano", "uf"], how="left")
base["confiavel"] = base["confiavel"].fillna(True).astype(bool)

todas = base.groupby("ano")["valor"].sum().rename("valor_todas_ufs")
ok = base[base["confiavel"]].groupby("ano")["valor"].sum().rename("valor_ufs_confiaveis")
n_ok = conf.groupby("ano")["confiavel"].sum().rename("ufs_confiaveis")
pop_ok = (
    conf[conf["confiavel"]]
    .merge(pop_uf_ano[["uf", "ano", "populacao"]], on=["uf", "ano"], how="left")
    .groupby("ano")["populacao"].sum().rename("pop_ufs_confiaveis")
)

serie_historica = pd.concat([todas, n_ok, ok, pop_ok], axis=1).reset_index()
serie_historica["parcial"] = serie_historica["ano"].map(MESES_ESPERADOS) < 12
serie_historica["taxa_100k_ufs_confiaveis"] = (
    serie_historica["valor_ufs_confiaveis"] / serie_historica["pop_ufs_confiaveis"] * 100_000
)
serie_historica.loc[serie_historica["parcial"], "taxa_100k_ufs_confiaveis"] = np.nan

display(serie_historica.round({"taxa_100k_ufs_confiaveis": 2}))

completos = serie_historica[~serie_historica["parcial"]]
parciais_s = serie_historica[serie_historica["parcial"]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(completos["ano"], completos["valor_todas_ufs"], "--o", color="gray", label="Todas as UFs")
ax1.plot(completos["ano"], completos["valor_ufs_confiaveis"], "-o", color="C0", label="Só UFs confiáveis")
if not parciais_s.empty:
    ax1.plot(parciais_s["ano"], parciais_s["valor_ufs_confiaveis"], "x", color="C3", markersize=10, label="Ano parcial")
ax1.set_title(f"Total nacional — {INDICADOR_ANALISE}")
ax1.set_xlabel("Ano")
ax1.set_ylabel("Total registrado")
ax1.grid(alpha=0.2)
ax1.legend()

ax2.plot(completos["ano"], completos["taxa_100k_ufs_confiaveis"], "-o", color="C2")
ax2.set_title("Taxa por 100 mil hab. (UFs confiáveis, anos completos)")
ax2.set_xlabel("Ano")
ax2.set_ylabel("Taxa por 100 mil")
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.show()

if ANOS_PARCIAIS:
    print("Atenção: anos parciais na série:", ANOS_PARCIAIS)

## 13. Distribuição mensal do indicador selecionado

In [ ]:
recorte = resumo_sinesp[
    (resumo_sinesp["ano"] == ANO_ANALISE)
    & (resumo_sinesp["indicador"] == INDICADOR_ANALISE)
].copy()

if ABRANGENCIA_ANALISE != "Todas":
    recorte = recorte[recorte["abrangencia"] == ABRANGENCIA_ANALISE].copy()

if recorte.empty:
    raise RuntimeError("Não existem registros para a seleção realizada.")

por_mes = (
    recorte.groupby(["mes_numero", "mes"], as_index=False)["valor"].sum()
    .sort_values("mes_numero")
)
display(por_mes)

ax = por_mes.plot(x="mes", y="valor", kind="line", marker="o", figsize=(11, 5), legend=False)
ax.set_title(f"Evolução mensal — {INDICADOR_ANALISE}, {rotulo_periodo(ANO_ANALISE)}")
ax.set_xlabel("Mês")
ax.set_ylabel("Total registrado")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 14. Análise por Unidade da Federação (total e taxa por 100 mil)

Esta tabela, `analise_uf`, é a base dos gráficos, outliers e mapas. Inclui:

- `ocorrencias` — total no ano/indicador/abrangência escolhidos;
- `confiavel` — falso quando a UF tem provável não-reporte (seção 5.2);
- `populacao` e `taxa_100_mil` — a taxa fica **`NaN` para UFs não confiáveis**, para não aparecerem como "as mais seguras".

In [ ]:
por_uf = (
    pd.DataFrame({"uf": sorted(UFS_VALIDAS)})
    .merge(recorte.groupby("uf", as_index=False)["valor"].sum().rename(columns={"valor": "ocorrencias"}), on="uf", how="left")
    .fillna({"ocorrencias": 0})
)

conf_ano = completude.loc[
    (completude["indicador"] == INDICADOR_ANALISE) & (completude["ano"] == ANO_ANALISE),
    ["uf", "meses_com_registro", "confiavel"],
]
analise_uf = por_uf.merge(conf_ano, on="uf", how="left")
analise_uf["confiavel"] = analise_uf["confiavel"].fillna(True).astype(bool)

analise_uf = analise_uf.merge(
    pop_uf_ano.loc[pop_uf_ano["ano"] == ANO_ANALISE, ["uf", "populacao"]], on="uf", how="left"
)
analise_uf["taxa_100_mil"] = analise_uf["ocorrencias"] / analise_uf["populacao"] * 100_000
analise_uf.loc[~analise_uf["confiavel"], "taxa_100_mil"] = np.nan
analise_uf["ano"] = ANO_ANALISE
analise_uf["indicador"] = INDICADOR_ANALISE

analise_uf = analise_uf.sort_values("ocorrencias", ascending=False).reset_index(drop=True)
display(analise_uf.round({"taxa_100_mil": 2}))

nao_conf = analise_uf.loc[~analise_uf["confiavel"], "uf"].tolist()
if nao_conf:
    print("UFs com dado suspeito (em cinza nos gráficos e sem taxa):", nao_conf)

d = analise_uf.sort_values("ocorrencias")
ax = d.plot(
    x="uf", y="ocorrencias", kind="barh", figsize=(10, 8), legend=False,
    color=["C0" if c else "lightgray" for c in d["confiavel"]],
)
ax.set_title(f"{INDICADOR_ANALISE} por UF — {rotulo_periodo(ANO_ANALISE)}\n(cinza = dados incompletos)")
ax.set_xlabel("Total registrado")
ax.set_ylabel("UF")
plt.tight_layout()
plt.show()

if analise_uf["taxa_100_mil"].notna().any():
    d = analise_uf.dropna(subset=["taxa_100_mil"]).sort_values("taxa_100_mil")
    ax = d.plot(x="uf", y="taxa_100_mil", kind="barh", figsize=(10, 8), legend=False)
    ax.set_title(f"Taxa por 100 mil habitantes — {INDICADOR_ANALISE}, {rotulo_periodo(ANO_ANALISE)}")
    ax.set_xlabel("Taxa por 100 mil habitantes")
    ax.set_ylabel("UF")
    plt.tight_layout()
    plt.show()
    if MESES_ESPERADOS[ANO_ANALISE] < 12:
        print("AVISO: ano parcial — a taxa é de um período menor que 12 meses.")
else:
    print("Taxa por 100 mil habitantes não calculada (sem população ou sem UFs confiáveis).")

## 15. Possíveis outliers entre as UFs

O IQR é calculado **apenas entre UFs confiáveis**. Em totais absolutos, os outliers tendem a refletir apenas o tamanho da população (ex.: SP); por isso o teste é repetido sobre a **taxa por 100 mil**, que é a comparação justa.

In [ ]:
def limites_iqr(serie):
    s = serie.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1, q3, iqr, q1 - 1.5 * iqr, q3 + 1.5 * iqr


validas = analise_uf[analise_uf["confiavel"]]
resultados_outlier = {}

for coluna in ["ocorrencias", "taxa_100_mil"]:
    if validas[coluna].notna().sum() < 4:
        continue
    q1, q3, iqr, lo, hi = limites_iqr(validas[coluna])
    resultados_outlier[coluna] = (lo, hi)
    analise_uf[f"outlier_{coluna}"] = (
        analise_uf["confiavel"]
        & ((analise_uf[coluna] < lo) | (analise_uf[coluna] > hi))
    )
    print(f"[{coluna}] Q1={q1:.2f}  Q3={q3:.2f}  IQR={iqr:.2f}  limite superior={hi:.2f}")
    display(analise_uf[analise_uf[f"outlier_{coluna}"]][["uf", coluna]].sort_values(coluna, ascending=False))

colunas_box = [c for c in ["ocorrencias", "taxa_100_mil"] if c in resultados_outlier]
if colunas_box:
    fig, eixos = plt.subplots(1, len(colunas_box), figsize=(5 * len(colunas_box), 6))
    eixos = np.atleast_1d(eixos)
    for ax, coluna in zip(eixos, colunas_box):
        validas[coluna].dropna().plot(kind="box", ax=ax)
        ax.set_title(f"{coluna}\n{INDICADOR_ANALISE}, {rotulo_periodo(ANO_ANALISE)}")
    plt.tight_layout()
    plt.show()

## 16. Análise por município

Para economizar memória, esta etapa recarrega **somente a planilha do ano selecionado**. O gráfico mostra os 30 municípios com maior valor; a tabela `por_municipio` contém todos.

Só funciona para indicadores **com detalhamento municipal** (seção 8.1). Para os demais, o notebook informa e segue adiante.

> Os valores são **absolutos**: municípios grandes dominam o topo. Taxas municipais exigiriam população por município (limitação registrada na seção 21).

In [ ]:
por_municipio = pd.DataFrame(columns=["uf", "municipio", "ocorrencias"])

n_mun = granularidade.loc[
    (granularidade["ano"] == ANO_ANALISE) & (granularidade["indicador"] == INDICADOR_ANALISE),
    "municipios_distintos",
]
n_mun = int(n_mun.iloc[0]) if len(n_mun) else 0

if n_mun < 100:
    print(f"'{INDICADOR_ANALISE}' em {ANO_ANALISE} NÃO tem detalhamento municipal "
          f"({n_mun} município(s) distinto(s)); a base só o traz por UF.")
    print("Indicadores com detalhamento municipal:", INDICADORES_COM_MUNICIPIO)
else:
    print(f"Carregando dados municipais de {ANO_ANALISE}...")
    df_municipios = ler_ano(ANO_ANALISE, incluir_municipio=True)
    df_municipios = df_municipios[df_municipios["indicador"] == INDICADOR_ANALISE].copy()
    print("Registros do indicador:", len(df_municipios))

    if ABRANGENCIA_ANALISE != "Todas":
        df_municipios = df_municipios[df_municipios["abrangencia"] == ABRANGENCIA_ANALISE].copy()
        print("Registros após filtro de abrangência:", len(df_municipios))

    df_municipios = df_municipios[
        ~df_municipios["municipio"].str.upper().isin(MUNICIPIOS_INVALIDOS)
    ].copy()

    por_municipio = (
        df_municipios.groupby(["uf", "municipio"], as_index=False)["valor"].sum()
        .rename(columns={"valor": "ocorrencias"})
        .sort_values("ocorrencias", ascending=False)
        .reset_index(drop=True)
    )
    print("Municípios encontrados:", len(por_municipio))

    if por_municipio.empty:
        print("Sem dados municipais para esta seleção.")
    else:
        display(por_municipio)

        top = por_municipio.head(30).sort_values("ocorrencias").copy()
        top["municipio_uf"] = top["municipio"] + " - " + top["uf"]
        ax = top.plot(x="municipio_uf", y="ocorrencias", kind="barh", figsize=(11, 10), legend=False)
        ax.set_title(f"30 municípios com maior total — {INDICADOR_ANALISE}, {rotulo_periodo(ANO_ANALISE)}")
        ax.set_xlabel("Total registrado")
        ax.set_ylabel("Município")
        plt.tight_layout()
        plt.show()

    del df_municipios
    gc.collect()

## 17. Mapa temático do Brasil

UFs com dado incompleto (ou sem população) aparecem em **cinza — "Sem dados"**, em vez de entrar como taxa baixa.

In [ ]:
URL_MALHA_UF = (
    "https://geoftp.ibge.gov.br/organizacao_do_territorio/"
    "malhas_territoriais/malhas_municipais/municipio_2024/"
    "Brasil/BR_UF_2024.zip"
)

arquivo_zip = Path("BR_UF_2024.zip")
pasta_malha = Path("BR_UF_2024")

if not arquivo_zip.exists():
    print("Baixando malha do IBGE...")
    resposta = requests.get(URL_MALHA_UF, timeout=180)
    resposta.raise_for_status()
    arquivo_zip.write_bytes(resposta.content)

if not pasta_malha.exists():
    pasta_malha.mkdir(exist_ok=True)
    with zipfile.ZipFile(arquivo_zip, "r") as z:
        z.extractall(pasta_malha)

shapefiles = list(pasta_malha.rglob("*.shp"))
if not shapefiles:
    raise FileNotFoundError("Shapefile do IBGE não encontrado.")

ufs_geo = gpd.read_file(shapefiles[0])
ufs_geo.columns = [str(c).lower() for c in ufs_geo.columns]

if "sigla_uf" in ufs_geo.columns:
    ufs_geo = ufs_geo.rename(columns={"sigla_uf": "uf"})
elif "uf" not in ufs_geo.columns:
    if "cd_uf" in ufs_geo.columns:
        ufs_geo["uf"] = pd.to_numeric(ufs_geo["cd_uf"], errors="coerce").map(CODIGO_UF)
    else:
        raise RuntimeError("Não foi possível localizar a UF na malha.")

mapa_uf = ufs_geo.merge(
    analise_uf[["uf", "ocorrencias", "confiavel", "populacao", "taxa_100_mil"]],
    on="uf", how="left",
)

if mapa_uf["taxa_100_mil"].notna().any():
    VARIAVEL_MAPA = "taxa_100_mil"
    TITULO_MAPA = "taxa por 100 mil habitantes"
else:
    VARIAVEL_MAPA = "ocorrencias"
    TITULO_MAPA = "total registrado"
    mapa_uf.loc[~mapa_uf["confiavel"].fillna(True).astype(bool), "ocorrencias"] = np.nan

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

quantidade_classes = min(5, mapa_uf[VARIAVEL_MAPA].dropna().nunique())

opcoes = {
    "column": VARIAVEL_MAPA,
    "legend": True,
    "edgecolor": "white",
    "linewidth": 0.5,
    "ax": ax,
    "missing_kwds": {"color": "lightgray", "label": "Sem dados / incompleto"},
}

if quantidade_classes >= 2:
    opcoes["scheme"] = "quantiles"
    opcoes["k"] = quantidade_classes

mapa_uf.plot(**opcoes)

ax.set_title(f"{INDICADOR_ANALISE} — {TITULO_MAPA} ({rotulo_periodo(ANO_ANALISE)})", fontsize=14)
ax.axis("off")
plt.tight_layout()

arquivo_mapa = PASTA_RESULTADOS / "mapa_analise_uf.png"
plt.savefig(arquivo_mapa, dpi=200, bbox_inches="tight")
plt.show()

print("Mapa salvo em:", arquivo_mapa)

## 18. Mapa interativo

In [ ]:
mapa_wgs84 = mapa_uf.to_crs(epsg=4326).copy()

mapa_interativo = folium.Map(location=[-14.5, -52.5], zoom_start=4, tiles="CartoDB positron")

dados_coropleth = mapa_wgs84[["uf", VARIAVEL_MAPA]].dropna()

if not dados_coropleth.empty:
    folium.Choropleth(
        geo_data=mapa_wgs84[["uf", "geometry"]],
        data=dados_coropleth,
        columns=["uf", VARIAVEL_MAPA],
        key_on="feature.properties.uf",
        fill_opacity=0.7,
        line_opacity=0.3,
        nan_fill_color="lightgray",
        legend_name=f"{TITULO_MAPA.capitalize()} — {INDICADOR_ANALISE} ({rotulo_periodo(ANO_ANALISE)})",
    ).add_to(mapa_interativo)

campos = ["uf", "ocorrencias"]
aliases = ["UF:", "Total registrado:"]

if mapa_wgs84["populacao"].notna().any():
    campos.append("populacao")
    aliases.append("População:")

if mapa_wgs84["taxa_100_mil"].notna().any():
    campos.append("taxa_100_mil")
    aliases.append("Taxa por 100 mil:")

folium.GeoJson(
    mapa_wgs84[campos + ["geometry"]],
    style_function=lambda feature: {"fillOpacity": 0, "weight": 0.5},
    tooltip=folium.GeoJsonTooltip(fields=campos, aliases=aliases, localize=True),
).add_to(mapa_interativo)

arquivo_html = PASTA_RESULTADOS / "mapa_interativo.html"
mapa_interativo.save(arquivo_html)

print("Mapa interativo salvo em:", arquivo_html)
mapa_interativo

## 19. Validação e exportação dos dados  *(entrada da Sprint 2)*

Antes de exportar, o notebook **valida** o resumo (sem chaves duplicadas, só as 27 UFs, só os 10 indicadores, sem valores negativos). Se alguma validação falhar, nada é exportado.

Arquivos para levar à Sprint 2:

| Arquivo | Uso na Sprint 2 |
|---|---|
| `sinesp_resumo_2015_2026.csv` | **obrigatório** — base do painel UF × ano |
| `cobertura_2015_2026.csv` | opcional — identifica anos parciais (a Sprint 2 também detecta sozinha) |
| `populacao_uf_ano.csv` | **recomendado** — mesma população usada aqui |
| `completude_uf_ano_indicador.csv` | opcional — auditoria das UFs/anos suspeitos |

Os demais (`indicadores`, `qualidade`, `granularidade`, `analise_uf_selecionada`, `analise_municipios_selecionada`) servem à documentação e ao relatório.

In [ ]:
# ---- Validações (se alguma falhar, nada é exportado) ----
chaves = ["ano", "mes_numero", "uf", "indicador", "abrangencia"]
assert not resumo_sinesp.duplicated(chaves).any(), "Há chaves duplicadas no resumo."
assert resumo_sinesp["indicador"].nunique() == len(INDICADORES_ALVO), "Número de indicadores diferente de 10."
assert set(resumo_sinesp["uf"]) <= UFS_VALIDAS, "UF inválida no resumo."
assert (resumo_sinesp["valor"] >= 0).all(), "Valores negativos no resumo."
assert resumo_sinesp["valor"].notna().all(), "Valores nulos no resumo."
assert set(resumo_sinesp["ano"]) == set(anos_encontrados), "Anos do resumo diferem dos arquivos lidos."
print("Validações do resumo: OK")

arquivos = {
    "sinesp_resumo_2015_2026.csv": resumo_sinesp,
    "cobertura_2015_2026.csv": cobertura,
    "indicadores.csv": indicadores_gerais,
    "populacao_uf_ano.csv": pop_uf_ano,
    "completude_uf_ano_indicador.csv": completude,
    "qualidade_dados_por_ano.csv": qualidade,
    "granularidade_municipal.csv": granularidade,
    "analise_uf_selecionada.csv": analise_uf,
}

if not por_municipio.empty:
    arquivos["analise_municipios_selecionada.csv"] = por_municipio
else:
    print("(análise municipal vazia para esta seleção — arquivo de municípios não gerado)")

for nome, tabela in arquivos.items():
    tabela.to_csv(PASTA_PROCESSADOS / nome, index=False, encoding="utf-8-sig")

print("\nArquivos gerados em", PASTA_PROCESSADOS)
for nome in arquivos:
    print("-", nome)
print("-", PASTA_RESULTADOS / "mapa_analise_uf.png")
print("-", PASTA_RESULTADOS / "mapa_interativo.html")

## 20. Conclusões da EDA

Execute a célula abaixo: ela calcula automaticamente os principais achados a partir dos dados. Depois complemente com a sua interpretação.

In [ ]:
print("1) Anos disponíveis:", anos_encontrados)
print("   Anos parciais:", {a: f"até {MESES[MESES_ESPERADOS[a]]}" for a in ANOS_PARCIAIS} or "nenhum")

print("\n2) Indicadores encontrados em todos os anos:",
      bool((verificacao.reindex(list(INDICADORES_ALVO)) > 0).all().all()))

n_susp = int(completude["suspeito"].sum())
print(f"\n3) Combinações UF×ano×indicador (alto volume) com provável não-reporte: {n_susp}")
if n_susp:
    resumo_susp = (
        completude[completude["suspeito"]].groupby("ano")["uf"].nunique().rename("ufs_afetadas")
    )
    print("   UFs afetadas por ano:", resumo_susp.to_dict())

print("\n4) Indicadores só em nível de UF:", INDICADORES_SO_UF)

print(f"\n5) Seleção atual: {INDICADOR_ANALISE}, {rotulo_ano(ANO_ANALISE)}")
top3 = analise_uf[analise_uf["confiavel"]].nlargest(3, "ocorrencias")["uf"].tolist()
print("   Maiores totais (UFs confiáveis):", top3)
if analise_uf["taxa_100_mil"].notna().any():
    top3t = analise_uf.nlargest(3, "taxa_100_mil")["uf"].tolist()
    print("   Maiores taxas por 100 mil:", top3t)
    print("   O ranking por taxa difere do ranking por total?", top3 != top3t)

if "outlier_taxa_100_mil" in analise_uf:
    print("   Outliers pela taxa:", analise_uf.loc[analise_uf["outlier_taxa_100_mil"], "uf"].tolist() or "nenhum")

### Pontos para registrar no relatório

1. quais anos estão disponíveis e até qual mês (2026 é **parcial**);
2. se os 10 indicadores aparecem em todos os anos;
3. **quais UFs/anos têm provável não-reporte** e como isso foi tratado (seção 5.2);
4. como o indicador evoluiu **nas UFs confiáveis** (seção 12) — e quanto da variação do total bruto era cobertura;
5. quais UFs lideram em total **e** em taxa por 100 mil, e por que os rankings diferem;
6. quais indicadores só existem por UF;
7. possíveis outliers (pela taxa);
8. limitações encontradas.

## 21. Limitações

- Os arquivos anuais têm centenas de milhares de linhas e levam alguns minutos para processar.
- **Zero em indicador de alto volume ≠ ausência de crime.** A heurística da seção 5.2 (todos os meses com registro) é uma aproximação: pode marcar como suspeito um ano legítimo com um mês sem registro, ou deixar passar um não-reporte parcial. A lista `ufs_suspeitas` deve ser conferida contra as notas metodológicas do Sinesp.
- Indicadores de baixo volume (feminicídio, latrocínio, morte de agente etc.) **não** passam pelo teste de completude, e seus zeros podem esconder não-reporte, sobretudo nos primeiros anos (ex.: tentativa de feminicídio cresce muito após 2018, em parte por adoção do registro).
- A forma de registro varia entre as UFs; indicadores diferentes usam `total_vitima` ou `total` (seção 8). Quando a fração de linhas com `total_vitima` muda entre anos, a unidade de medida mudou.
- A variável `abrangencia` deve ser considerada na interpretação. Ao escolher "Todas", os valores das abrangências são **somados**; confirme na documentação do Sinesp que elas não se sobrepõem (não é possível verificar isso apenas pelos dados exportados).
- Alguns indicadores existem só em nível de UF (seção 8.1); a análise municipal não se aplica a eles. As contagens municipais são absolutas (sem população municipal).
- A população de 2023 é **interpolada** e a de anos nas pontas, repetida; 2022 vem do Censo. Há quebra metodológica entre as estimativas anteriores ao Censo 2022 e as posteriores, o que pode afetar levemente as taxas em torno de 2022–2024.
- A taxa de 2026 (ano parcial) não é comparável à de anos completos e não é calculada na série nacional.
- A análise é exploratória e não estabelece relação causal.
- Para os indicadores de natureza hedionda condicional (homicídio doloso, lesão corporal seguida de morte, morte de agente do Estado e tentativa de homicídio), a base não permite separar os casos hediondos dos demais.
- O código foi validado com dados sintéticos no formato do Sinesp VDE; as consultas reais ao SIDRA/IBGE (em especial a do Censo 2022, tabela 4714) devem ser conferidas na primeira execução com internet.